In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Embedding

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

# Replace with your actual file path
file_path = '/content/drive/MyDrive/conversation.xlsx'
df = pd.read_excel(file_path)

# Display the data
print(df.head())


                     Brushaski                           English
0             besn echaa ayaas         what are you doing sister
1              ja ayas bo khin                  she is my sister
2        ja xum jot bo ja ayas      my sister is younger than me
3               ja daltas ayas               my beautiful sister
4  ja ayas but samajhdaaran bo  my sister is really intelligent 


In [3]:
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Extract English and Brushaski columns
english_sentences = df['English'].astype(str).tolist()
brushaski_sentences = df['Brushaski'].astype(str).tolist()

# Function to clean sentences
def clean_text(sentences):
    cleaned_sentences = []
    for sentence in sentences:
        sentence = sentence.lower()  # Convert to lowercase
        sentence = re.sub(r'\n', ' ', sentence)  # Remove newline characters
        sentence = re.sub(r'[^\w\s]', '', sentence)  # Remove punctuation
        sentence = re.sub(r'\s+', ' ', sentence)  # Replace multiple spaces with a single space
        sentence = sentence.strip()  # Remove leading/trailing spaces
        cleaned_sentences.append(sentence)
    return cleaned_sentences

# Apply cleaning function
english_sentences = clean_text(english_sentences)
brushaski_sentences = clean_text(brushaski_sentences)

# Print a few cleaned sentences to verify
print("Sample cleaned English sentence:", english_sentences[:1])
print("Sample cleaned Brushaski sentence:", brushaski_sentences[:1])


Sample cleaned English sentence: ['what are you doing sister']
Sample cleaned Brushaski sentence: ['besn echaa ayaas']


In [4]:
# Tokenize English sentences
english_tokenizer = Tokenizer()
english_tokenizer.fit_on_texts(english_sentences)
english_sequences = english_tokenizer.texts_to_sequences(english_sentences)
english_vocab_size = len(english_tokenizer.word_index) + 1

# Tokenize Brushaski sentences
start_token = "<start>"
brushaski_tokenizer = Tokenizer()
brushaski_tokenizer.fit_on_texts(brushaski_sentences)
brushaski_sequences = brushaski_tokenizer.texts_to_sequences(brushaski_sentences)
brushaski_vocab_size = len(brushaski_tokenizer.word_index) + 1

# Pad sequences to the same length
max_english_len = max(len(seq) for seq in english_sequences)
max_brushaski_len = max(len(seq) for seq in brushaski_sequences)

english_padded = pad_sequences(english_sequences, maxlen=max_english_len, padding='post')
brushaski_padded = pad_sequences(brushaski_sequences, maxlen=max_brushaski_len, padding='post')

print(f"English Vocab Size: {english_vocab_size}, Brushaski Vocab Size: {brushaski_vocab_size}")

English Vocab Size: 4239, Brushaski Vocab Size: 9910


In [5]:
print(english_padded[0])
print(brushaski_padded[0])

[ 39  32   5 181 386   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0]
[1228 2585 1229    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   

In [6]:
#data preparation
import numpy as np

# Decoder target data (Brushaski sequences shifted by one step)
decoder_target_data = np.zeros_like(brushaski_padded)
decoder_target_data[:, :-1] = brushaski_padded[:, 1:]

# Expand dimensions to match sparse categorical cross-entropy format
decoder_target_data = decoder_target_data[..., np.newaxis]


In [7]:
english_vocab_size = len(english_tokenizer.word_index) + 1  # +1 for padding
brushaski_vocab_size = len(brushaski_tokenizer.word_index) + 1


In [8]:
# Display a few tokens and their words
english_word_to_token = english_tokenizer.word_index  # Mapping: word -> token
english_token_to_word = {v: k for k, v in english_word_to_token.items()}  # Reverse mapping: token -> word

# Print first 10 tokens and their words
print("English Tokens (word -> token):")
for word, token in list(english_word_to_token.items())[:10]:
    print(f"{word}: {token}")

print("\nEnglish Tokens (token -> word):")
for token, word in list(english_token_to_word.items())[:10]:
    print(f"{token}: {word}")


English Tokens (word -> token):
the: 1
he: 2
and: 3
to: 4
you: 5
that: 6
a: 7
was: 8
i: 9
of: 10

English Tokens (token -> word):
1: the
2: he
3: and
4: to
5: you
6: that
7: a
8: was
9: i
10: of


In [9]:
# Display a few tokens and their words
brushaski_word_to_token = brushaski_tokenizer.word_index  # Mapping: word -> token
brushaski_token_to_word = {v: k for k, v in brushaski_word_to_token.items()}  # Reverse mapping: token -> word

# Print first 10 tokens and their words
print("Brushaski Tokens (word -> token):")
for word, token in list(brushaski_word_to_token.items())[:10]:
    print(f"{word}: {token}")

print("\nBrushaski Tokens (token -> word):")
for token, word in list(brushaski_token_to_word.items())[:10]:
    print(f"{token}: {word}")


Brushaski Tokens (word -> token):
ke: 1
ine: 2
ne: 3
baa: 4
bam: 5
cum: 6
daa: 7
ǰaa: 8
besan: 9
kaa: 10

Brushaski Tokens (token -> word):
1: ke
2: ine
3: ne
4: baa
5: bam
6: cum
7: daa
8: ǰaa
9: besan
10: kaa


In [10]:
# 2. Split the dataset into train and validation sets
from sklearn.model_selection import train_test_split

# Split into train (80%) and validation (20%) sets
english_train, english_val, brushaski_train, brushaski_val = train_test_split(
    english_padded, brushaski_padded, test_size=0.2, random_state=42
)

decoder_target_train, decoder_target_val = train_test_split(
    decoder_target_data, test_size=0.2, random_state=42
)

# Print dataset shapes
print(f"Training samples: {english_train.shape[0]}")
print(f"Validation samples: {english_val.shape[0]}")



Training samples: 2502
Validation samples: 626


In [11]:
# 3. Build the model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

encoder_inputs = Input(shape=(None,), name="encoder_inputs")
encoder_embedding = Embedding(input_dim=english_vocab_size, output_dim=256, mask_zero=True)(encoder_inputs)
encoder_lstm = LSTM(256, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None,), name="decoder_inputs")
decoder_embedding = Embedding(input_dim=brushaski_vocab_size, output_dim=256, mask_zero=True)(decoder_inputs)
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
decoder_dense = Dense(brushaski_vocab_size, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs            │ (None, None)           │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ decoder_inputs            │ (None, None)           │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding (Embedding)     │ (None, None, 256)      │      1,085,184 │ encoder_inputs[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ not_equal (NotEqual)      │ (None, None)           │              0 │ encoder_inputs[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_1 (Embedding)   │ (None, None, 256)      │      2,536,960 │ decoder_inputs[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm (LSTM)               │ [(None, 256), (None,   │        525,312 │ embedding[0][0],       │
│                           │ 256), (None, 256)]     │                │ not_equal[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_1 (LSTM)             │ [(None, None, 256),    │        525,312 │ embedding_1[0][0],     │
│                           │ (None, 256), (None,    │                │ lstm[0][1], lstm[0][2] │
│                           │ 256)]                  │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, None, 9910)     │      2,546,870 │ lstm_1[0][0]           │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 7,219,638 (27.54 MB)

 Trainable params: 7,219,638 (27.54 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
model.fit(
    [english_train, brushaski_train],
    decoder_target_train,
    batch_size=32,
    epochs=200,  # Increase epochs
    validation_data=([english_val, brushaski_val], decoder_target_val)
)


Epoch 1/200
79/79 ━━━━━━━━━━━━━━━━━━━━ 7s 83ms/step - accuracy: 0.9930 - loss: 0.0097 - val_accuracy: 0.8990 - val_loss: 11.3033
Epoch 2/200
79/79 ━━━━━━━━━━━━━━━━━━━━ 6s 82ms/step - accuracy: 0.9940 - loss: 0.0093 - val_accuracy: 0.8991 - val_loss: 11.3200
Epoch 3/200
79/79 ━━━━━━━━━━━━━━━━━━━━ 10s 82ms/step - accuracy: 0.9939 - loss: 0.0099 - val_accuracy: 0.8996 - val_loss: 11.3372
Epoch 4/200
79/79 ━━━━━━━━━━━━━━━━━━━━ 10s 82ms/step - accuracy: 0.9944 - loss: 0.0094 - val_accuracy: 0.8994 - val_loss: 11.3551
Epoch 5/200
79/79 ━━━━━━━━━━━━━━━━━━━━ 6s 81ms/step - accuracy: 0.9935 - loss: 0.0084 - val_accuracy: 0.8994 - val_loss: 11.3695
Epoch 6/200
79/79 ━━━━━━━━━━━━━━━━━━━━ 10s 80ms/step - accuracy: 0.9943 - loss: 0.0092 - val_accuracy: 0.8997 - val_loss: 11.3832
Epoch 7/200
79/79 ━━━━━━━━━━━━━━━━━━━━ 10s 80ms/step - accuracy: 0.9938 - loss: 0.0087 - val_accuracy: 0.8998 - val_loss: 11.4058
Epoch 8/200
79/79 ━━━━━━━━━━━━━━━━━━━━ 10s 81ms/step - accuracy: 0.9946 - loss: 0.0084 - val_

In [41]:
model.save("brushaski_translator_model.h5")
print("Model saved successfully!")


Model saved successfully!


In [42]:
# Fit tokenizer
brushaski_tokenizer = Tokenizer(filters='')  # Avoid removing any special tokens
brushaski_tokenizer.fit_on_texts(brushaski_sentences)

# Ensure the tokenizer includes <start> and <end> tokens
start_token = "<start>"
if start_token not in brushaski_tokenizer.word_index:
    brushaski_tokenizer.word_index[start_token] = len(brushaski_tokenizer.word_index) + 1
    brushaski_tokenizer.index_word[len(brushaski_tokenizer.word_index)] = start_token

end_token = "<end>"
if end_token not in brushaski_tokenizer.word_index:
    brushaski_tokenizer.word_index[end_token] = len(brushaski_tokenizer.word_index) + 1
    brushaski_tokenizer.index_word[len(brushaski_tokenizer.word_index)] = end_token

# Save their indices for later use
start_token_index = brushaski_tokenizer.word_index[start_token]
end_token_index = brushaski_tokenizer.word_index[end_token]


In [43]:
# Encoder model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder model
decoder_state_input_h = Input(shape=(256,))
decoder_state_input_c = Input(shape=(256,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_embedding, initial_state=decoder_states_inputs
)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states
)


In [44]:
def decode_sequence(input_seq):
    # Encode the input sequence to get the initial states
    states_value = encoder_model.predict(input_seq)

    # Prepare the target sequence with the <start> token
    target_seq = np.zeros((1, 1))  # Shape (batch_size=1, 1 timestep)
    target_seq[0, 0] = start_token_index  # Start token index

    stop_condition = False
    decoded_sentence = ""
    output_length = 0  # Track output length
    max_decoder_len = 30

    while not stop_condition:
        # Predict the next token and states using the decoder model
        output_tokens, state_h, state_c = decoder_model.predict(
            [target_seq] + states_value
        )

        # Sample the next word token (greedy decoding)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = brushaski_tokenizer.index_word.get(sampled_token_index, "")

        # Append the word to the decoded sentence (if not <end>)
        if sampled_word != "<end>":
            decoded_sentence += " " + sampled_word

        # Check stop condition
        if sampled_word == "<end>" or output_length >= max_decoder_len:
            stop_condition = True

        # Update the target sequence with the sampled token
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # Update states for the next time step
        states_value = [state_h, state_c]

        output_length += 1

    return decoded_sentence.strip()


In [79]:
sample_sentence = "thank you"
sample_sequence = english_tokenizer.texts_to_sequences([sample_sentence.lower()])
sample_padded = pad_sequences(sample_sequence, maxlen=max_english_len, padding='post')

# Generate translation
translation = decode_sequence(sample_padded)

# Display the input and output
print(f"English: {sample_sentence}")
print(f"Brushaski: {translation}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━